In [4]:
import pandas as pd

# 1. Load the previously filtered dataset
input_file = 'E:/Projects/BTP/Data/Raw_data/INDIA/Final_data.csv'
df = pd.read_csv(input_file)

# Combine the split date columns into a proper Pandas Datetime object
df['Date'] = pd.to_datetime(
    df[['year', 'mon', 'day']].rename(columns={'mon': 'month'})
)

# 2. Aggregate duplicate reports for the same district on the same date
df_agg = df.groupby(
    ['state_ut', 'district', 'Date']
)[['Cases', 'Deaths']].sum().reset_index()

# 3. Build the Master Calendar using the cleaned, aggregated data
master_calendar = pd.date_range(
    start=df_agg['Date'].min(),
    end=df_agg['Date'].max(),
    freq='W-MON'
)

# 4. Create the MultiIndex guaranteeing every district gets a row per week
districts = df_agg[['state_ut', 'district']].drop_duplicates()

multi_idx = pd.MultiIndex.from_product(
    [districts['state_ut'], districts['district'], master_calendar],
    names=['state_ut', 'district', 'Date']
)

# 5. Reindex and Zero-Fill safely
df_indexed = df_agg.set_index(
    ['state_ut', 'district', 'Date']
)

df_continuous = df_indexed.reindex(multi_idx).reset_index()

df_continuous['Cases'] = df_continuous['Cases'].fillna(0)
df_continuous['Deaths'] = df_continuous['Deaths'].fillna(0)

# Restore disease name for all rows, including newly created rows
df_continuous['Disease'] = 'Acute Diarrhoeal Disease'

# 6. Save to a new file
output_file = 'E:/Projects/BTP/Data/Clean/INDIA/phase2_diarrhea_continuous_zerofilled.csv'
df_continuous.to_csv(output_file, index=False)

print("✅ Success! Duplicates merged and fragmented timeline repaired.")
print(f"✅ New version saved to: {output_file}")

MemoryError: Unable to allocate 3.24 GiB for an array with shape (434201175,) and data type object

In [5]:
import pandas as pd

# 1. Load the previously filtered dataset
input_file = 'E:/Projects/BTP/Data/Raw_data/INDIA/Final_data.csv'
df = pd.read_csv(input_file)

# Combine the split date columns into a proper Pandas Datetime object
df['Date'] = pd.to_datetime(
    df[['year', 'mon', 'day']].rename(columns={'mon': 'month'})
)

# 2. Aggregate duplicate reports for the same district on the same date
df_agg = df.groupby(
    ['state_ut', 'district', 'Date']
)[['Cases', 'Deaths']].sum().reset_index()

# 3. Build the Master Calendar using the cleaned, aggregated data
master_calendar = pd.date_range(
    start=df_agg['Date'].min(),
    end=df_agg['Date'].max(),
    freq='W-MON'
)

# 4. Create the MultiIndex guaranteeing every district gets a row per week
# Get actual state-district combinations
districts = df_agg[['state_ut', 'district']].drop_duplicates()

# Add every week to every ACTUAL state-district combination
districts['key'] = 1
calendar_df = pd.DataFrame({'Date': master_calendar, 'key': 1})

multi_idx = (
    districts
    .merge(calendar_df, on='key')
    .drop(columns='key')
)

multi_idx = pd.MultiIndex.from_frame(multi_idx)
multi_idx.names = ['state_ut', 'district', 'Date']
# 5. Reindex and Zero-Fill safely
df_indexed = df_agg.set_index(
    ['state_ut', 'district', 'Date']
)

df_continuous = df_indexed.reindex(multi_idx).reset_index()

df_continuous['Cases'] = df_continuous['Cases'].fillna(0)
df_continuous['Deaths'] = df_continuous['Deaths'].fillna(0)

# Restore disease name for all rows, including newly created rows
df_continuous['Disease'] = 'Acute Diarrhoeal Disease'

# 6. Save to a new file
output_file = 'E:/Projects/BTP/Data/Clean/INDIA/phase2_diarrhea_continuous_zerofilled.csv'
df_continuous.to_csv(output_file, index=False)

print("✅ Success! Duplicates merged and fragmented timeline repaired.")
print(f"✅ New version saved to: {output_file}")

✅ Success! Duplicates merged and fragmented timeline repaired.
✅ New version saved to: E:/Projects/BTP/Data/Clean/INDIA/phase2_diarrhea_continuous_zerofilled.csv
